In [ ]:
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

In [56]:
df = pd.read_excel("Pulso_clientes.xlsx" , sheet_name = "clientes")

In [57]:

df.head(5)

,id_cliente,fecha_registro,ciudad,edad,plan,canal_adquisicion,precio_mensual_cop,dispositivo,completo_onboarding,sesiones_primer_mes,sesiones_ultimos_30d,fecha_ultima_sesion,tickets_soporte,nps,estado,fecha_cancelacion
0,C1001,2025-11-30 00:00:00,Bucaramanga,29,Anual,Referido,19900,iOS,No,1,11,2026-09-26,0,6.0,Activo,NaT
1,C1002,2026-06-09 00:00:00,Bogotá,18,Mensual,Promo 3 meses,29900,iOS,Sí,8,0,2026-08-30,1,NaN,Cancelado,2026-09-14
2,C1003,2026-01-16 00:00:00,Bogotá,40,Mensual,Referido,29900,iOS,Sí,12,7,2026-09-25,0,10.0,Activo,NaT
3,C1004,2026-07-13 00:00:00,Cali,39,Mensual,Redes sociales,29900,Android,No,1,1,2026-09-19,1,NaN,Cancelado,2026-09-21
4,C1005,2026-07-11 00:00:00,Cali,27,Mensual,Referido,29900,Android,Sí,15,6,2026-09-27,0,NaN,Activo,NaT


In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_cliente            306 non-null    str           
 1   fecha_registro        306 non-null    object        
 2   ciudad                306 non-null    str           
 3   edad                  306 non-null    int64         
 4   plan                  306 non-null    str           
 5   canal_adquisicion     306 non-null    str           
 6   precio_mensual_cop    306 non-null    int64         
 7   dispositivo           306 non-null    str           
 8   completo_onboarding   306 non-null    str           
 9   sesiones_primer_mes   306 non-null    int64         
 10  sesiones_ultimos_30d  306 non-null    int64         
 11  fecha_ultima_sesion   306 non-null    datetime64[us]
 12  tickets_soporte       306 non-null    int64         
 13  nps                   138 non-n

In [59]:
df["nps_completo"] = df["nps"].notna().astype(int)  # 1 = respondió el NPS, 0 = no respondió

In [60]:
df["fecha_registro"] =  pd.to_datetime(df["fecha_registro"])

In [61]:
df["duración"] = df["fecha_cancelacion"] - df["fecha_registro"]

In [ ]:
# ---------- Limpieza y validaciones ----------
FECHA_CORTE = pd.Timestamp("2026-09-30")
EDAD_MAXIMA = 100
SESIONES_MAXIMAS_30D = 100  # más de 100 sesiones en 30 días (>3 por día) se considera ilógico

# 1) estado: normalizar variantes ("Cancelada", "cancelado", "CANCELADO", ...)
estado_norm = df["estado"].str.strip().str.lower().map(
    lambda s: "Activo" if s.startswith("activ") else ("Cancelado" if s.startswith("cancel") else None)
)
print("Valores originales de estado:", df["estado"].value_counts().to_dict())
print("Valores de estado sin clasificar:", estado_norm.isna().sum())
df["estado"] = estado_norm

# 2) ciudad: estandarizar nombres (mayúsculas/minúsculas, tildes, espacios, "D.C.")
ciudades_validas = {"bogota": "Bogotá", "medellin": "Medellín", "cali": "Cali",
                    "barranquilla": "Barranquilla", "bucaramanga": "Bucaramanga"}


def clave_ciudad(texto):
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return texto.lower().replace("d.c.", "").strip()


ciudad_norm = df["ciudad"].map(lambda c: ciudades_validas.get(clave_ciudad(c)))
print("\nValores originales de ciudad:", df["ciudad"].value_counts().to_dict())
print("Valores de ciudad sin clasificar:", ciudad_norm.isna().sum())
df["ciudad"] = ciudad_norm.fillna(df["ciudad"])  # los no reconocidos se conservan tal cual
print("Ciudad normalizada:", df["ciudad"].value_counts().to_dict())

# 3) edad: valores ilógicos (> 100) se marcan como nulos
edad_ilogica = df["edad"] > EDAD_MAXIMA
print(f"\nEdades > {EDAD_MAXIMA} (ilógicas) marcadas como nulas: {edad_ilogica.sum()}")
if edad_ilogica.any():
    display(df.loc[edad_ilogica, ["id_cliente", "edad", "estado"]])
df.loc[edad_ilogica, "edad"] = None
print("Edades nulas:", df["edad"].isna().sum(), "| edad máxima:", df["edad"].max())

# 4) sesiones_ultimos_30d: valores ilógicos (> SESIONES_MAXIMAS_30D) se marcan como nulos
sesiones_ilogicas = df["sesiones_ultimos_30d"] > SESIONES_MAXIMAS_30D
print(f"\nsesiones_ultimos_30d > {SESIONES_MAXIMAS_30D} (ilógicas) marcadas como nulas: {sesiones_ilogicas.sum()}")
if sesiones_ilogicas.any():
    display(df.loc[sesiones_ilogicas, ["id_cliente", "sesiones_ultimos_30d", "sesiones_primer_mes", "estado"]])
df.loc[sesiones_ilogicas, "sesiones_ultimos_30d"] = None
print("Sesiones nulas:", df["sesiones_ultimos_30d"].isna().sum(),
      "| máximo:", df["sesiones_ultimos_30d"].max())

# 5) IDs duplicados: se eliminan (se conserva la primera aparición de cada ID)
dup_ids = df["id_cliente"].duplicated(keep=False)
cols_sin_derivadas = [c for c in df.columns if not c.startswith("flag_")]
n_identicas = df.duplicated(subset=cols_sin_derivadas).sum()
n_dup_filas = df["id_cliente"].duplicated().sum()
print(f"\nIDs duplicados: {df.loc[dup_ids, 'id_cliente'].nunique()} IDs; se eliminan {n_dup_filas} filas "
      f"({n_identicas} eran idénticas en todas las columnas)")
if n_dup_filas != n_identicas:
    print("¡Atención! Hay IDs repetidos con datos distintos, se conserva solo la primera fila:")
    display(df[dup_ids].sort_values("id_cliente"))
df = df.drop_duplicates(subset="id_cliente", keep="first").reset_index(drop=True)
print("Filas tras eliminar duplicados:", len(df))

# 6) Activos con fecha_cancelacion -> se marcan como Cancelado
activos_con_cancelacion = (df["estado"] == "Activo") & df["fecha_cancelacion"].notna()
print(f"\nActivos con fecha_cancelacion reclasificados a Cancelado: {activos_con_cancelacion.sum()}")
if activos_con_cancelacion.any():
    display(df.loc[activos_con_cancelacion, ["id_cliente", "estado", "fecha_registro",
                                              "fecha_ultima_sesion", "fecha_cancelacion"]])
df.loc[activos_con_cancelacion, "estado"] = "Cancelado"
df["estado_binario"] = (df["estado"] == "Cancelado").astype(int)  # 1 = Cancelado, 0 = Activo
print("Estado final:", df["estado"].value_counts().to_dict())

# 7) Fechas posteriores a la fecha de corte (30-sep-2026): fecha_ultima_sesion se limita a la fecha de corte
post_corte_sesion = df["fecha_ultima_sesion"] > FECHA_CORTE
print(f"\nfecha_ultima_sesion posterior al corte: {post_corte_sesion.sum()} casos -> se fija en {FECHA_CORTE.date()}")
if post_corte_sesion.any():
    display(df.loc[post_corte_sesion, ["id_cliente", "estado", "fecha_registro",
                                       "fecha_ultima_sesion", "fecha_cancelacion"]])
df.loc[post_corte_sesion, "fecha_ultima_sesion"] = FECHA_CORTE
df["flag_ultima_sesion_post_corte"] = df["fecha_ultima_sesion"] > FECHA_CORTE
df["flag_cancelacion_post_corte"] = df["fecha_cancelacion"] > FECHA_CORTE

# 8) Inconsistencias entre fechas (comparar con NaT da False, así que los activos no se marcan)
df["flag_registro_post_ultima_sesion"] = df["fecha_registro"] > df["fecha_ultima_sesion"]
df["flag_registro_post_cancelacion"] = df["fecha_registro"] > df["fecha_cancelacion"]
df["flag_ultima_sesion_post_cancelacion"] = df["fecha_ultima_sesion"] > df["fecha_cancelacion"]

flags = [c for c in df.columns if c.startswith("flag_")]
print("\nResumen de flags (número de casos):")
print(df[flags].sum().to_string())

cols_fechas = ["id_cliente", "estado", "fecha_registro", "fecha_ultima_sesion", "fecha_cancelacion"]
for flag in flags:
    casos = df[df[flag]]
    print(f"\n{flag}: {len(casos)} casos")
    if len(casos):
        display(casos.sort_values("id_cliente")[cols_fechas])

# 9) Consistencia entre estado y fecha_cancelacion
print("\nCancelados sin fecha_cancelacion:", ((df["estado"] == "Cancelado") & df["fecha_cancelacion"].isna()).sum())
print("Activos con fecha_cancelacion:", ((df["estado"] == "Activo") & df["fecha_cancelacion"].notna()).sum())

In [ ]:
# ---------- Corrección de fecha_cancelacion anterior a fecha_registro ----------
# Se reemplaza por fecha_ultima_sesion + promedio de días entre la última sesión y la cancelación
# de los clientes con el mismo estado cuya fecha_cancelacion sí es válida
# (no es anterior ni a fecha_registro ni a fecha_ultima_sesion). Estos 3 casos son todos "Cancelado".
invalidos = (df["fecha_cancelacion"] < df["fecha_registro"]) | (df["fecha_cancelacion"] < df["fecha_ultima_sesion"])
validos = df["fecha_cancelacion"].notna() & ~invalidos
dias_post_sesion = (df["fecha_cancelacion"] - df["fecha_ultima_sesion"])[validos].dt.days
prom_post_sesion = dias_post_sesion.groupby(df["estado"]).mean()
print("Promedio de días entre última sesión y cancelación, por estado (solo casos válidos):")
print(prom_post_sesion.round(1).to_string())

antes = df.loc[invalidos, ["id_cliente", "estado", "fecha_registro", "fecha_ultima_sesion", "fecha_cancelacion"]].copy()
dias = df.loc[invalidos, "estado"].map(prom_post_sesion).round().astype(int)
df.loc[invalidos, "fecha_cancelacion"] = df.loc[invalidos, "fecha_ultima_sesion"] + pd.to_timedelta(dias, unit="D")

# Recalcular variables derivadas y flags afectados
df["duración"] = df["fecha_cancelacion"] - df["fecha_registro"]
df["flag_registro_post_cancelacion"] = df["fecha_registro"] > df["fecha_cancelacion"]
df["flag_ultima_sesion_post_cancelacion"] = df["fecha_ultima_sesion"] > df["fecha_cancelacion"]
df["flag_cancelacion_post_corte"] = df["fecha_cancelacion"] > FECHA_CORTE

antes["fecha_cancelacion_nueva"] = df.loc[invalidos, "fecha_cancelacion"]
antes["dias_tras_ultima_sesion"] = dias
print(f"\nCasos corregidos: {invalidos.sum()}")
display(antes.rename(columns={"fecha_cancelacion": "fecha_cancelacion_original"}))
print("Duraciones negativas restantes:", (df["duración"] < pd.Timedelta(0)).sum())
print("flag_registro_post_cancelacion:", df["flag_registro_post_cancelacion"].sum(),
      "| flag_ultima_sesion_post_cancelacion:", df["flag_ultima_sesion_post_cancelacion"].sum(),
      "| flag_cancelacion_post_corte:", df["flag_cancelacion_post_corte"].sum())

In [ ]:
sns.set_theme(style="whitegrid")
colores_estado = {"Activo": "#2a9d8f", "Cancelado": "#e76f51"}

# Distribución de estado
fig, ax = plt.subplots(figsize=(5, 4))
conteo = df["estado"].value_counts()
ax.bar(conteo.index, conteo.values, color=[colores_estado[e] for e in conteo.index])
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v} ({v / len(df):.1%})", ha="center", va="bottom")
ax.set_title("Distribución de estado de los clientes")
ax.set_xlabel("Estado")
ax.set_ylabel("Número de clientes")
ax.set_ylim(0, conteo.max() * 1.12)
plt.tight_layout()
plt.show()

In [ ]:
# Variables categóricas: gráfico general + prevalencia de estado por categoría
def graficar_categorica(col):
    orden = df[col].value_counts().index
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.suptitle(f"Variable categórica: {col}", fontsize=14, fontweight="bold")

    # General
    conteo = df[col].value_counts().loc[orden]
    axes[0].bar(conteo.index.astype(str), conteo.values, color="#457b9d")
    for i, v in enumerate(conteo.values):
        axes[0].text(i, v, str(v), ha="center", va="bottom")
    axes[0].set_title("Distribución general")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Número de clientes")
    axes[0].set_ylim(0, conteo.max() * 1.12)

    # Prevalencia de estado por categoría (% dentro de cada categoría)
    prev = pd.crosstab(df[col], df["estado"], normalize="index").loc[orden] * 100
    prev = prev[[e for e in ["Activo", "Cancelado"] if e in prev.columns]]
    prev.plot(kind="bar", stacked=True, ax=axes[1],
              color=[colores_estado[e] for e in prev.columns])
    for cont in axes[1].containers:
        axes[1].bar_label(cont, fmt="%.0f%%", label_type="center", color="white")
    axes[1].set_title("Prevalencia de estado por categoría")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("% de clientes dentro de la categoría")
    axes[1].set_ylim(0, 100)
    axes[1].legend(title="Estado", loc="upper left", bbox_to_anchor=(1, 1))

    for ax in axes:
        ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()


for col in ["ciudad", "plan", "canal_adquisicion", "dispositivo",
            "completo_onboarding", "nps_completo"]:
    graficar_categorica(col)

In [ ]:
# Variables numéricas: distribución general, entre Activos y entre Cancelados
df["duración_dias"] = df["duración"].dt.days  # solo existe para clientes cancelados
# Inactividad: días entre la última sesión y la cancelación (solo existe para clientes cancelados)
df["inactividad_dias"] = (df["fecha_cancelacion"] - df["fecha_ultima_sesion"]).dt.days

etiquetas = {"duración_dias": "duración (días entre registro y cancelación)",
             "inactividad_dias": "inactividad (días entre última sesión y cancelación)",
             "precio_mensual_cop": "precio mensual (COP)"}

# Formato legible de los ejes, p. ej. 29900 -> $29.900
formatos = {"precio_mensual_cop": FuncFormatter(lambda x, _: f"${x:,.0f}".replace(",", "."))}


def graficar_numerica(col):
    # Mismos ejes X e Y en los tres paneles para poder compararlos directamente
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True, sharey=True)
    fig.suptitle(f"Variable numérica: {etiquetas.get(col, col)}", fontsize=14, fontweight="bold")
    grupos = [("General", df, "#457b9d"),
              ("Estado = Activo", df[df["estado"] == "Activo"], colores_estado["Activo"]),
              ("Estado = Cancelado", df[df["estado"] == "Cancelado"], colores_estado["Cancelado"])]
    bins = min(30, max(df[col].nunique(), 5))
    bordes = np.histogram_bin_edges(df[col].dropna(), bins=bins)  # mismos bins en los tres paneles
    for ax, (nombre, sub, color) in zip(axes, grupos):
        datos = sub[col].dropna()
        if datos.empty:
            ax.text(0.5, 0.5, "Sin datos", ha="center", va="center", transform=ax.transAxes)
        else:
            sns.histplot(datos, bins=bordes, color=color, ax=ax)
        ax.set_title(f"{nombre} (n={len(datos)})")
        ax.set_xlabel(etiquetas.get(col, col))
        ax.set_ylabel("Número de clientes")
        ax.tick_params(labelbottom=True, labelleft=True)
        if col in formatos:
            # Una marca por cada valor existente, con etiquetas legibles y en diagonal hacia abajo
            valores = np.sort(df[col].dropna().unique())
            ax.set_xticks(valores)
            ax.set_xticklabels([formatos[col](v, None) for v in valores],
                               rotation=45, ha="right", rotation_mode="anchor")
    plt.tight_layout()
    plt.show()


for col in ["edad", "precio_mensual_cop", "sesiones_primer_mes",
            "sesiones_ultimos_30d", "tickets_soporte", "duración_dias", "inactividad_dias"]:
    graficar_numerica(col)

In [ ]:
# ---------- Comentarios de clientes ----------
dfc = pd.read_excel("Pulso_comentarios.xlsx", sheet_name="comentarios")
print("Comentarios:", dfc.shape)
print(dfc["sentimiento"].value_counts().to_dict())
display(dfc.head())

# Solo comentarios con sentimiento definido (se excluyen spam / no relacionados: "No aplica")
dfc_s = dfc[dfc["sentimiento"].isin(["Positivo", "Negativo"])].copy()
print(f"\nComentarios usados en los gráficos: {len(dfc_s)} (excluidos 'No aplica': {len(dfc) - len(dfc_s)})")

In [ ]:
# Sentimiento de los comentarios por subcategoría
colores_sent = {"Positivo": "#2a9d8f", "Negativo": "#e76f51"}

# 1) Distribución general del sentimiento
fig, ax = plt.subplots(figsize=(5, 4))
conteo = dfc_s["sentimiento"].value_counts().reindex(["Positivo", "Negativo"])
ax.bar(conteo.index, conteo.values, color=[colores_sent[s] for s in conteo.index])
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v} ({v / len(dfc_s):.0%})", ha="center", va="bottom")
ax.set_title("Distribución del sentimiento de los comentarios")
ax.set_xlabel("Sentimiento")
ax.set_ylabel("Número de comentarios")
ax.set_ylim(0, conteo.max() * 1.12)
plt.tight_layout()
plt.show()

# 2) y 3) Por subcategoría: número de comentarios y % de cada sentimiento
tabla = pd.crosstab(dfc_s["subcategoria"], dfc_s["sentimiento"]).reindex(columns=["Positivo", "Negativo"], fill_value=0)
tabla["total"] = tabla.sum(axis=1)
tabla = tabla.sort_values("total")  # de menor a mayor para que la más frecuente quede arriba
pct = tabla[["Positivo", "Negativo"]].div(tabla["total"], axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle("Sentimiento de los comentarios por subcategoría", fontsize=14, fontweight="bold")

tabla[["Positivo", "Negativo"]].plot(kind="barh", stacked=True, ax=axes[0],
                                     color=[colores_sent[s] for s in ["Positivo", "Negativo"]])
for cont in axes[0].containers:
    axes[0].bar_label(cont, labels=[int(v) if v > 0 else "" for v in cont.datavalues],
                      label_type="center", color="white", fontweight="bold")
axes[0].set_title("Número de comentarios")
axes[0].set_xlabel("Número de comentarios")
axes[0].set_ylabel("Subcategoría")
axes[0].get_legend().remove()

pct.plot(kind="barh", stacked=True, ax=axes[1], color=[colores_sent[s] for s in ["Positivo", "Negativo"]])
for cont in axes[1].containers:
    axes[1].bar_label(cont, labels=[f"{v:.0f}%" if v > 0 else "" for v in cont.datavalues],
                      label_type="center", color="white", fontweight="bold")
axes[1].set_title("% de comentarios dentro de la subcategoría")
axes[1].set_xlabel("% de comentarios")
axes[1].set_ylabel("Subcategoría")
axes[1].set_xlim(0, 100)
axes[1].tick_params(labelleft=True)
axes[1].legend(title="Sentimiento", loc="upper left", bbox_to_anchor=(1, 1))  # fuera del gráfico
plt.tight_layout()
plt.show()